# Candle Prediction using Market Depth

In [496]:
from pathlib import Path
import pandas as pd
import numpy as np
from utils import resample_fractional_minute, apply_trailing_logic

In [497]:
# ---- Input ------
date_ = "23APR2026"
file_name = "NIFTY26APR24200CE.xlsx"

file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}")
df = pd.read_excel(file_path)
if "PE" in file_name or "CE" in file_name:
    col_name = "last_trade_time"

    # Volume computation
    # 1. Calculate the basic difference between rows
    df["volume_momentary"] = df["volume_traded"].diff()
    df.loc[df["volume_momentary"] == 0, "volume_momentary"] = np.nan
    df["volume_momentary"] = df["volume_momentary"].ffill()
    df["volume_momentary"] = df["volume_momentary"].fillna(0)
else:
    col_name = "local_time"

df[col_name] = pd.to_datetime(df[col_name])
target_date = pd.to_datetime(date_).date()
df = df[df[col_name].dt.date == target_date]

In [498]:
file_name

'NIFTY26APR24200CE.xlsx'

In [499]:
df.head(4)

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,last_traded_quantity,average_traded_price,option_CE_PE,option_type,...,total_sell_quantity,ohlc,change,oi,oi_day_high,oi_day_low,depth,tradable,mode,volume_momentary
2,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0
3,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0
4,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0
5,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0


In [500]:
# ---- Input ------
N = 6
window = 3
threshold = 0.002 # price pct change pct

In [501]:
df.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_momentary'],
      dtype='str')

In [502]:
print(type(df.iloc[0]["local_time"]))
print(df.iloc[0]["local_time"])
print(df.iloc[0]["last_trade_time"])

<class 'pandas.Timestamp'>
2026-04-23 09:15:00.619000
2026-04-23 09:15:00


In [503]:
clubbed_df = resample_fractional_minute(df, col_name, N)
clubbed_df["bucket_time_next"] = clubbed_df["bucket_time"].shift(-1)
clubbed_df["price_diff"] = clubbed_df["close"] - clubbed_df["open"]
clubbed_df["price_pct"] = (clubbed_df["close"] - clubbed_df["open"])/clubbed_df["open"]

clubbed_df["volume_diff"] = clubbed_df["volume_close"] - clubbed_df["volume_open"]
clubbed_df["volume_pct"] = (clubbed_df["volume_close"] - clubbed_df["volume_open"])/clubbed_df["volume_open"]


In [504]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].head()

,bucket_time,minute,open,high,low,close,volume_clubbed,volume_open,volume_high,volume_low,...,candle_type,minute_open,minute_high,minute_low,minute_close,bucket_time_next,price_diff,price_pct,volume_diff,volume_pct
270,2026-04-23 10:00:00,2026-04-23 10:00:00,243.20,243.65,241.00,243.65,34710.0,10010.0,10010.0,2015.0,...,SELL,243.2,243.65,235.65,235.8,2026-04-23 10:00:10,0.45,0.001850,-1170.0,-0.116883
271,2026-04-23 10:00:10,2026-04-23 10:00:00,242.30,243.60,241.25,241.25,22230.0,5070.0,5525.0,1820.0,...,SELL,243.2,243.65,235.65,235.8,2026-04-23 10:00:20,-1.05,-0.004333,-2795.0,-0.551282
272,2026-04-23 10:00:20,2026-04-23 10:00:00,241.95,242.90,241.10,241.10,20475.0,3965.0,6630.0,1430.0,...,SELL,243.2,243.65,235.65,235.8,2026-04-23 10:00:30,-0.85,-0.003513,2665.0,0.672131
273,2026-04-23 10:00:30,2026-04-23 10:00:00,241.55,241.80,238.60,240.20,28405.0,1690.0,9620.0,1235.0,...,SELL,243.2,243.65,235.65,235.8,2026-04-23 10:00:40,-1.35,-0.005589,2990.0,1.769231
274,2026-04-23 10:00:40,2026-04-23 10:00:00,239.65,239.65,236.10,236.10,74555.0,2665.0,39195.0,2275.0,...,SELL,243.2,243.65,235.65,235.8,2026-04-23 10:00:50,-3.55,-0.014813,3640.0,1.365854


In [505]:
clubbed_df[["volume_open", "volume_high", "volume_low", "volume_close", "volume_clubbed"]]

,volume_open,volume_high,volume_low,volume_close,volume_clubbed
0,15665.0,118235.0,15665.0,74360.0,556855.0
1,68380.0,81900.0,40625.0,40625.0,362310.0
2,46735.0,46735.0,30810.0,30810.0,230490.0
3,40690.0,89635.0,20995.0,33410.0,269815.0
4,51935.0,53885.0,25935.0,26260.0,228215.0
...,...,...,...,...,...
2245,24505.0,27755.0,12285.0,12285.0,147420.0
2246,19565.0,24960.0,15405.0,18980.0,113555.0
2247,25870.0,25870.0,7800.0,14690.0,94770.0
2248,17485.0,17485.0,7215.0,11115.0,68705.0


In [506]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].shape

(36, 21)

In [580]:
def generate_signal(
    df,
    window: int,
    threshold: float,
    volume_threshold: float = 0,

    use_price_pct_level=False,
    use_price_trend=True,
    use_volume=False
):

    print("use_price_pct_level : ", use_price_pct_level)
    print("use_price_trend : ", use_price_trend)
    print("use_volume : ", use_volume)
    # ---------------------------
    # BASE CONDITIONS (level)
    # ---------------------------


    if use_price_pct_level:
        cond_buy = df["price_pct"] > threshold
        cond_sell = df["price_pct"] < -threshold
        buy_threshold_streak = cond_buy.rolling(window).min() == 1
        sell_threshold_streak = cond_sell.rolling(window).min() == 1
    else:
        buy_threshold_streak = pd.Series(True, index=df.index)
        sell_threshold_streak = pd.Series(True, index=df.index)

    # ---------------------------
    # PRICE TREND (monotonic)
    # ---------------------------
    if use_price_trend:
        price_diff = df["price_diff"].diff()

        cond_buy_trend = price_diff > 0
        cond_sell_trend = price_diff < 0

        buy_trend_streak = cond_buy_trend.rolling(window).min() == 1
        sell_trend_streak = cond_sell_trend.rolling(window).min() == 1
    else:
        buy_trend_streak = pd.Series(True, index=df.index)
        sell_trend_streak = pd.Series(True, index=df.index)

    # ---------------------------
    # VOLUME CONDITION
    # ---------------------------
    if use_volume:
        print("use_volume : ", use_volume)
        cond_vol = (df["volume_diff"] > volume_threshold)
        vol_streak = cond_vol.rolling(window).min() == 1
    else:
        vol_streak = pd.Series(True, index=df.index)

    # ---------------------------
    # FINAL STREAKS
    # ---------------------------
    buy_streak = buy_threshold_streak & buy_trend_streak & vol_streak
    sell_streak = sell_threshold_streak & sell_trend_streak & vol_streak

    # ---------------------------
    # SIGNAL
    # ---------------------------
    df["predicted"] = np.where(
        buy_streak, "BUY",
        np.where(sell_streak, "SELL", None)
    )
    return df

In [581]:
clubbed_df_2 = generate_signal(clubbed_df, window, threshold, 1000, use_price_pct_level=True, use_price_trend=True, use_volume=False)

use_price_pct_level :  True
use_price_trend :  True
use_volume :  False


In [582]:
clubbed_df_2[clubbed_df_2["predicted"]==clubbed_df["candle_type"]].head(5)

,bucket_time,minute,open,high,low,close,volume_clubbed,volume_open,volume_high,volume_low,...,minute_open,minute_high,minute_low,minute_close,bucket_time_next,price_diff,price_pct,volume_diff,volume_pct,predicted
11,2026-04-23 09:16:50,2026-04-23 09:16:00,191.00,196.85,190.00,196.85,244270.0,33735.0,51675.0,31070.0,...,182.80,196.85,177.10,196.85,2026-04-23 09:17:00,5.85,0.030628,8450.0,0.250482,BUY
12,2026-04-23 09:17:00,2026-04-23 09:17:00,195.55,204.30,195.55,202.00,398905.0,73255.0,92430.0,38545.0,...,195.55,204.30,195.55,197.20,2026-04-23 09:17:10,6.45,0.032984,-17680.0,-0.241349,BUY
77,2026-04-23 09:27:50,2026-04-23 09:27:00,196.55,199.20,196.05,198.70,75530.0,10985.0,26260.0,10465.0,...,194.95,199.20,192.50,198.70,2026-04-23 09:28:00,2.15,0.010939,5460.0,0.497041,BUY
78,2026-04-23 09:28:00,2026-04-23 09:28:00,198.95,205.20,198.95,205.20,225355.0,17290.0,67730.0,17290.0,...,198.95,207.15,198.95,205.90,2026-04-23 09:28:10,6.25,0.031415,9880.0,0.571429,BUY
96,2026-04-23 09:31:00,2026-04-23 09:31:00,211.00,214.50,210.15,214.50,308945.0,39845.0,107965.0,36270.0,...,211.00,222.85,210.15,213.65,2026-04-23 09:31:10,3.50,0.016588,68120.0,1.709625,BUY


In [583]:
# df_with_signal = df.merge(
#         clubbed_df[["bucket_time", "predicted"]],
#         left_on="last_trade_time",
#         right_on="bucket_time",
#         how="left"
#     )

import pandas as pd

# 1. Ensure both DataFrames are sorted by the time columns
df = df.sort_values("last_trade_time")
clubbed_df_2 = clubbed_df_2.sort_values("bucket_time")

# 2. Perform the proximity merge
df_with_signal = pd.merge_asof(
    df,
    clubbed_df_2[["bucket_time", "predicted"]],
    left_on="last_trade_time",
    right_on="bucket_time",
    direction="backward" # Only looks at the past/current, never the future
)

# Find duplicates in bucket_time and set their 'predicted' value to NaN
df_with_signal.loc[df_with_signal.duplicated(subset=['bucket_time'], keep='first'), 'predicted'] = np.nan

In [584]:
# df_with_signal.to_excel("df_with_signal.xlsx")

In [585]:
# clubbed_df_2.to_excel("clubbed_df_2.xlsx")

In [586]:
df_with_signal.head()

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,last_traded_quantity,average_traded_price,option_CE_PE,option_type,...,change,oi,oi_day_high,oi_day_low,depth,tradable,mode,volume_momentary,bucket_time,predicted
0,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
1,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
2,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
3,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
4,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:02.120,2026-04-23 09:15:01,184.05,130,191.93,CE,atm,...,-43.818681,1252485,1252485,1252485,"{'buy': [{'quantity': 910, 'price': 182.55, 'o...",True,full,105430.0,2026-04-23 09:15:00,NaN


In [587]:
df_with_signal.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_momentary', 'bucket_time', 'predicted'],
      dtype='str')

In [588]:
params = {
    "initial_sl_pct": 0.02,
    "target_pct": 0.01,
    "trail_sl_pct": 0.02,
    "tight_sl_offset": 0.5,
}

In [589]:
trades = apply_trailing_logic(df_with_signal, params)

trades = pd.DataFrame(trades)
if len(trades):
    trades["final"] = trades.apply(lambda row: "profit" if row["profit"] > 0 else "loss", axis=1)
else:
    print("trades not generated")


In [590]:
len(trades)

18

In [591]:
# trades

In [592]:
trades[trades["final"]=="profit"]["profit"].sum()

np.float64(46.299999999999955)

In [593]:
trades[trades["final"]=="loss"]["profit"].sum()

np.float64(-2.0260000000000105)

In [594]:
trades.head(11)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,final
0,2026-04-23 09:16:51,191.00,2026-04-23 09:17:01,196.350,5.350,0.028010,profit
1,2026-04-23 09:27:51,196.55,2026-04-23 09:27:58,198.700,2.150,0.010939,profit
2,2026-04-23 09:28:00,198.95,2026-04-23 09:28:11,205.500,6.550,0.032923,profit
3,2026-04-23 09:31:00,210.35,2026-04-23 09:31:11,215.050,4.700,0.022344,profit
4,2026-04-23 09:53:00,243.10,2026-04-23 09:53:08,246.650,3.550,0.014603,profit
5,2026-04-23 09:53:10,246.80,2026-04-23 09:53:17,249.500,2.700,0.010940,profit
6,2026-04-23 10:08:20,243.00,2026-04-23 10:08:34,245.500,2.500,0.010288,profit
7,2026-04-23 10:16:31,230.55,2026-04-23 10:16:36,233.700,3.150,0.013663,profit
8,2026-04-23 10:25:50,225.65,2026-04-23 10:25:55,227.500,1.850,0.008199,profit
9,2026-04-23 11:08:10,194.40,2026-04-23 11:09:34,192.374,-2.026,-0.010422,loss


In [595]:
trades["final"].value_counts()

final
profit    17
loss       1
Name: count, dtype: int64

In [573]:
trades[trades["final"]=="loss"].head(12)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,final


In [526]:

# clubbed_df_2.to_excel(Path(f"assets/logs/{date_}/clubbed_df.xlsx"))

In [527]:
clubbed_df_2["predicted"].value_counts()

predicted
BUY     20
SELL    14
Name: count, dtype: int64